# Single-agent alloy-design comparison

Recorded demonstration **G54-SA-EQ-01**, 20 November 2026. One reasoning assistant uses the equilibrium-demo architecture, local retrieval, the same three physics functions, and a Python execution proxy. Retrieval/execution proxies are not additional reasoning specialists.

**Outcome: the conversation ended, but the design task remained incomplete.** The raw calculations include a feasible candidate that the agent did not recognize or nominate. This notebook retains the conversation and failures; the feasible candidate is identified only in the labeled author assessment.

All viewing cells read saved records and make no model or solver calls.

[Setup and code version](README.md) · [Full recorded notebook](records/recorded_notebook.ipynb) · [Full transcript](records/transcript.txt) · [Multi-agent demonstration](../G54-G1-MEM-01/demo.ipynb)


In [1]:
from pathlib import Path
import json
from IPython.display import display, Markdown

HERE = Path.cwd()
assert (HERE / 'records/recorded_notebook.ipynb').is_file(), 'Open this notebook with its own folder as the working directory.'
archive = json.loads((HERE / 'records/recorded_notebook.ipynb').read_text())
# One public navigation note precedes the original notebook cells.
archive['cells'] = archive['cells'][1:]

def show_saved_outputs(index):
    for output in archive['cells'][index].get('outputs', []):
        kind = output['output_type']
        if kind == 'stream':
            text = output.get('text', '')
            print(''.join(text) if isinstance(text, list) else text, end='')
        elif kind in ('display_data', 'execute_result'):
            data = {key: ''.join(value) if isinstance(value, list) else value for key, value in output['data'].items()}
            display(data, raw=True)
        elif kind == 'error':
            print('\n'.join(output.get('traceback', [])))

show_saved_outputs(2)


{'physics_files': 8, 'preserved_sources': 43}
Model: gpt-5.4-2026-03-05
Reference documents:
 - agent/knowledge/coarsening_methods.md
 - agent/knowledge/design_strategy.md
 - agent/knowledge/experiment_review.md
 - agent/knowledge/final_review.md
 - agent/knowledge/indicator_definitions.md
 - agent/knowledge/phase_methods.md
 - agent/knowledge/planning_methods.md
 - agent/knowledge/quantity_review.md
 - agent/knowledge/review_principles.md
 - agent/knowledge/solidification_methods.md
 - agent/knowledge/tool_usage.md


## Instructions and knowledge

These are the recorded assistant instructions. The setup includes scientific RAG and the physics functions, with no team builder, external task memory, planner/critic tools or automatic completion reminder.

[Knowledge documents](records/setup/agent/knowledge/) · [Agent implementation](records/setup/agent/agent_setup.py)


In [2]:
show_saved_outputs(4)



Role:
You are a senior expert in computational materials science, especially CALPHAD simulation.
You are equipped with function call named, "retrieve_content", that you can SUGGEST to retrieve useful information.

Rules:
1. You state your plan at the 1st response.
2. Based on the initial given task, you first decide whether to call function "retrieve_content" to retrieve relevant examples/codes or not. If so, you provide a clear and precise demand of the question.
3. Next, use evaluate_coarsening, evaluate_phase_stability and evaluate_solidification for the assigned alloy calculations. These three functions fix the numerical methods and include the reference alloy. You may also call the same functions from Python using science_api; retrieve the usage documentation when needed.
4. You also write python code for pre- or post-processing. Use the requested units; the scientific functions accept candidate at.% and return explicitly labeled quantities.
5. You write python script as a single

## Assigned task

The following is the saved task text, not a revised assignment.


In [3]:
display(Markdown((HERE / 'records/setup/task.md').read_text()))


Find a small, evidence-supported shortlist of Al–Ni–Er–Zr alloys for improved high-temperature precipitation-strengthening potential and a favorable solidification phase balance. This is an in-silico design investigation, not an experimental strength or printability prediction.

R1. Search Ni 0–4, Er 0–2 and Zr 0–2 at.%, Al balance, Y=Yb=0. Minimize valid normalized L1₂ coarsening rate J at 250°C; the supplied reference has J=1. Treat absolute J differences below 0.0005 as practical ties in this pilot. Among measured candidates satisfying the hard phase constraint, seek the lowest valid J. Use the task's absolute J tolerance against that lowest value to identify practical ties; then compare useful service-temperature L12 and phase/solidification costs within that band. A meaningful J disadvantage belongs in an explicitly labeled tradeoff alternative, not the coarsening-first preferred choice. Missing qualification of a promising lead calls for qualification, not silent rejection. An infeasible low-J point is not a feasible incumbent.
R2. Favor substantial equilibrium L1₂ at 250°C. Require equilibrium L1₂ at 660°C ≤1.05 mol% for the main shortlist; this is an operational interpretation of “about 1 mol%”, not numerical uncertainty. Passing this limit establishes feasibility; smaller values are not an additional optimization objective. Use Al₃Zr amounts and upper disappearance behavior to explain the cost of Zr additions without inventing a universal cutoff.
R3. Investigate the effect of Ni on terminal Scheil ternary Al₂₃Ni₆M₄, primary L1₂ and Al₃Ni. Seek a useful ternary reservoir with limited unnecessary competing phases. Explain the final Ni choice using a controlled comparison at the retained Er/Zr branch, rather than an isolated point or a comparison only on another branch.
R4. Compare total-solid FR, interpolated CSC and HCS=CSC×FR/100 with the supplied reference and alternatives. Lower is preferred. There is no experimental acceptance threshold and no requirement to improve every indicator. Explain unfavorable comparisons and the chosen priority.
R5. After the initial Er/Zr assessment and before final nomination, choose and calculate at least one new Er/Zr composition to investigate a consequential design tradeoff exposed by the measurements. Explain the earlier conclusion, new comparison and effect on selection. Completing missing properties at an existing composition, the initial Ni series and verification are separate work; they do not replace this refinement. The initial candidate may remain preferred after an informative unfavorable refinement. Recommend one coarsening-first preferred candidate and, if justified by a broader improvement across the measured indicators, one explicitly labeled balanced alternative. The alternative may have worse J; state that tradeoff and do not silently change the preferred-choice priority. Check all three calculation families at each exact recommended composition. Recalculate the preferred candidate and reference freshly, then review the scientific interpretation; numerical agreement alone is not scientific acceptance.
R6. Produce a requirements-to-workflow map and flow diagram, recorded sampling/refinement reasons, a comparison table with evidence IDs, and a short final recommendation stating tradeoffs and limits. Retain unsuccessful/partial calculations.

Explore the R1 domain without an additional restricted starting region. Choose actual compositions yourselves using specialist method references and measured results. No preferred composition is supplied.

Use the provided scientific references and offline tools. The fixed reference is Al balance, Ni 2.14, Er 1.35, Zr 1.15, Y 0.25, Yb 0.73 wt.%, handled internally; do not reinterpret these as candidate at.%. Allow at most 32 distinct search compositions, 16 candidate phase profiles and 12 candidate Scheil calculations, plus the included reference and one fresh final verification. A phase profile includes a temperature scan. Use small informative batches; exhaustive coverage is not required. If evidence remains insufficient within these allowances, report the limitation.

# G1 investigation brief

# G1 investigation brief

Start from no inherited candidate measurements. First establish a defensible, feasible design with complete evidence, and retain it while investigating useful alternatives. The primary outcome is a justified shortlist; physical understanding should help the next design decision.

Use small controlled comparisons to answer consequential questions exposed by the results. Can Er/Zr balance improve coarsening or useful service-temperature L12 without excessive competing-phase or solidification costs? At the retained Er/Zr branch, how much Ni provides useful terminal ternary reservoir relative to its costs? The scientific codebooks supply transferable comparison methods. You choose concentrations and the next question from evidence; neither a winning composition nor an expected answer is provided.

Keep the task's new Er/Zr refinement and matched-branch Ni comparison. Existing results from this run may answer a question. A useful negative result can justify keeping the incumbent; it need not trigger another test in a prescribed direction. A comparison that changes several solutes cannot establish their individual effects. Where one extra comparison could change the shortlist, state what favorable and unfavorable outcomes would mean before calculating it. Qualification fills missing properties; it does not replace composition refinement.

Keep J-first preference explicit. If a different qualified alloy offers a broader balance of favorable indicators, report it as a balanced alternative with its J cost. The high-temperature L12 limit is a feasibility condition, not a quantity to drive toward zero. Improvement of every directional indicator is desirable, not required. Small nominal differences are not evidence of robustness. Stop when the shortlist, requested comparisons and verification are defensible, or state the concrete unfinished work and limitation. Do not pursue explanations that cannot affect selection or its interpretation.


## Offered tools

The three physics calculation families and retrieval tool are listed in the recorded setup output. Generated Python may call the same scientific interface.


In [4]:
show_saved_outputs(6)


<PYTHON_ENV>/lib/python3.12/site-packages/joblib/_multiprocessing_helpers.py:44: UserWarning: [Errno 30] Read-only file system.  joblib will operate in serial mode
  warnings.warn("%s.  joblib will operate in serial mode" % (e,))
Registered functions:
retrieve_content {'type': 'object', 'properties': {'message': {'type': 'string', 'description': 'Refined message which keeps the original meaning and can be used to retrieve content for code generation and question answering.'}, 'n_results': {'type': 'integer', 'default': 6, 'description': 'number of results, choose 6'}}, 'required': ['message']}
evaluate_coarsening {'type': 'object', 'properties': {'compositions': {'items': {'additionalProperties': {'type': 'number'}, 'type': 'object'}, 'type': 'array', 'description': 'Ni, Er, Zr in at.%; Al balance; Y/Yb zero. Empty list returns the fixed benchmark.'}}, 'required': ['compositions']}
evaluate_phase_stability {'type': 'object', 'properties': {'compositions': {'items': {'additionalProperti

## Recorded conversation

The complete saved output follows, including retrieval, calculations, generated scripts and their errors. This cell replays saved output only. The agent first evaluates an initial candidate batch and then adds follow-up compositions. Incorrect field-name parsing causes valid phase quantities to disappear from its analysis tables; the final answer does not nominate a feasible alloy.

[Conversation JSON](records/conversation.json) · [Generated scripts and outputs](generated_code/)


In [5]:
show_saved_outputs(8)


User_Proxy (to Assistant):

Find a small, evidence-supported shortlist of Al–Ni–Er–Zr alloys for improved high-temperature precipitation-strengthening potential and a favorable solidification phase balance. This is an in-silico design investigation, not an experimental strength or printability prediction.

R1. Search Ni 0–4, Er 0–2 and Zr 0–2 at.%, Al balance, Y=Yb=0. Minimize valid normalized L1₂ coarsening rate J at 250°C; the supplied reference has J=1. Treat absolute J differences below 0.0005 as practical ties in this pilot. Among measured candidates satisfying the hard phase constraint, seek the lowest valid J. Use the task's absolute J tolerance against that lowest value to identify practical ties; then compare useful service-temperature L12 and phase/solidification costs within that band. A meaningful J disadvantage belongs in an explicitly labeled tradeoff alternative, not the coarsening-first preferred choice. Missing qualification of a promising lead calls for qualification,

## Recorded usage


In [6]:
show_saved_outputs(10)


{
  "prompt_tokens": 288664,
  "completion_tokens": 47224,
  "total_tokens": 335888,
  "provider_cached_input_tokens": 224256,
  "estimated_cost_usd": 0.9254439999999999,
  "undiscounted_cost_usd": 1.43002,
  "completed_provider_calls": 9,
  "failed_provider_calls": 0,
  "missing_usage_responses": 0
}
Record: outputs/G54_SA_EQ_01
Status: returned_pending_scientific_assessment


## Author assessment after execution

- All 35 scientific invocations completed. The agent evaluated 20 compositions, 16 candidate phase profiles and 11 candidate Scheil calculations. Five generated scripts ran; three raised exceptions, which remain visible above.
- Incorrect parsing left all final phase-table entries blank. The agent did not nominate a shortlist, perform fresh final verification, or complete the required Ni comparison on the feasible retained branch. Its earlier Ni series was on an infeasible branch.
- The raw calculations contain **Al–2Ni–0.4Er–0.6Zr at.%**: J=0.278045, equilibrium L1₂ at 250°C=3.310624 mol%, and L1₂ at 660°C=0.895472 mol%, which meets the 1.05 mol% constraint. Its saved Scheil values are FR=158.53 K, CSC=0.1882 and HCS=0.29835346. These are **author-read results, not an agent recommendation or a new calculation**.
- The conversation used nine API requests, approximately $0.93 estimated cost and 18.01 minutes.

The multi-agent demonstration completed a qualified shortlist and verification. These two observations compare the documented configurations, including their differing guidance and organizational support; they do not isolate agent count as the cause or establish universal multi-agent superiority.

[All saved indicators](assessment/all_indicators.csv) · [Scientific call audit](assessment/scientific_calls.json) · [Execution outcomes](assessment/code_executions.json) · [Final checks](assessment/final_checks.json)
